In [1]:
import pandas as pd
import folium
import branca.colormap as cm
import matplotlib.pyplot as plt
import os

os.makedirs('WYNIKI/MAPY', exist_ok=True)
os.makedirs('WYNIKI/WYKRESY', exist_ok=True)

def stworz_mape(trasy_df, stacje_df, nazwa_pliku):
    mapa = folium.Map(location=[40.720, -74.040], zoom_start=14)
    min_val = trasy_df['liczba_tras'].min()
    max_val = trasy_df['liczba_tras'].max()
    skala_kolorow = cm.LinearColormap(
        colors=['#ff9999', '#ff0000', '#8b0000'],
        vmin=min_val, vmax=max_val,
        caption='Liczba przejazdów',
    )
    skala_kolorow.add_to(mapa)
    
    uzywane_stacje = set(trasy_df['start_station_name']).union(set(trasy_df['end_station_name']))
    for nazwa in uzywane_stacje:
        stacja = stacje_df[stacje_df['nazwa_stacji'] == nazwa]
        if not stacja.empty:
            folium.Marker(
                location=[stacja.iloc[0]['szerokosc_geo'], stacja.iloc[0]['dlugosc_geo']],
                popup=nazwa,
                icon=folium.Icon(color='blue', icon='bicycle', prefix='fa')
            ).add_to(mapa)
            
    for _, row in trasy_df.iterrows():
        start = stacje_df[stacje_df['nazwa_stacji'] == row['start_station_name']]
        koniec = stacje_df[stacje_df['nazwa_stacji'] == row['end_station_name']]
        if not start.empty and not koniec.empty:
            folium.PolyLine(
                locations=[
                    [start.iloc[0]['szerokosc_geo'], start.iloc[0]['dlugosc_geo']],
                    [koniec.iloc[0]['szerokosc_geo'], koniec.iloc[0]['dlugosc_geo']]
                ],
                color=skala_kolorow(row['liczba_tras']),
                weight=max(1, row['liczba_tras'] / 75),
                opacity=0.8,
                tooltip=f"{row['start_station_name']} -> {row['end_station_name']} ({row['liczba_tras']} przejazdów)"
            ).add_to(mapa)
    mapa.save(nazwa_pliku)

def stworz_wykres_top_10(trasy_df, tytul, sciezka_zapisu):
    df_top = trasy_df.sort_values(by='liczba_tras', ascending=False).head(10).copy()
    df_top['trasa'] = df_top['start_station_name'] + " - " + df_top['end_station_name']
    plt.figure(figsize=(12, 6))
    plt.barh(df_top['trasa'], df_top['liczba_tras'], color='skyblue')
    plt.xlabel('Liczba przejazdów')
    plt.ylabel('Trasa')
    plt.title(tytul)
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.savefig(sciezka_zapisu)
    plt.close()

def przetworz_miesiac(sciezka_pliku):
    df = pd.read_csv(sciezka_pliku)
    trasy = df.groupby(['start_station_name', 'end_station_name']).size().reset_index(name='liczba_tras')
    trasy['stacja_1'] = trasy[['start_station_name', 'end_station_name']].min(axis=1)
    trasy['stacja_2'] = trasy[['start_station_name', 'end_station_name']].max(axis=1)
    trasy = trasy.groupby(['stacja_1', 'stacja_2'])['liczba_tras'].sum().reset_index()
    trasy = trasy.rename(columns={'stacja_1': 'start_station_name', 'stacja_2': 'end_station_name'})
    return trasy.sort_values(by='liczba_tras', ascending=False).head(100).reset_index(drop=True)

trasy_styczen = przetworz_miesiac('DANE/JC-202501-citibike-tripdata.csv')
trasy_kwiecien = przetworz_miesiac('DANE/JC-202504-citibike-tripdata.csv')
trasy_lipiec = przetworz_miesiac('DANE/JC-202507-citibike-tripdata.csv')
trasy_pazdziernik = przetworz_miesiac('DANE/JC-202510-citibike-tripdata.csv')

wszystkie_dane = pd.concat([
    pd.read_csv('DANE/JC-202501-citibike-tripdata.csv'), 
    pd.read_csv('DANE/JC-202504-citibike-tripdata.csv'),
    pd.read_csv('DANE/JC-202507-citibike-tripdata.csv'),
    pd.read_csv('DANE/JC-202510-citibike-tripdata.csv')
])
stacje_start = wszystkie_dane[['start_station_name', 'start_lat', 'start_lng']].drop_duplicates().rename(columns={'start_station_name': 'nazwa_stacji', 'start_lat': 'szerokosc_geo', 'start_lng': 'dlugosc_geo'})
stacje_end = wszystkie_dane[['end_station_name', 'end_lat', 'end_lng']].drop_duplicates().rename(columns={'end_station_name': 'nazwa_stacji', 'end_lat': 'szerokosc_geo', 'end_lng': 'dlugosc_geo'})
stacje = pd.concat([stacje_start, stacje_end]).drop_duplicates(subset=['nazwa_stacji'])

lista_danych = [
    (trasy_styczen, 'WYNIKI/MAPY/mapa_styczen.html', 'WYNIKI/WYKRESY/wykres_styczen.png', 'Styczeń 2025'),
    (trasy_kwiecien, 'WYNIKI/MAPY/mapa_kwiecien.html', 'WYNIKI/WYKRESY/wykres_kwiecien.png', 'Kwiecień 2025'),
    (trasy_lipiec, 'WYNIKI/MAPY/mapa_lipiec.html', 'WYNIKI/WYKRESY/wykres_lipiec.png', 'Lipiec 2025'),
    (trasy_pazdziernik, 'WYNIKI/MAPY/mapa_pazdziernik.html', 'WYNIKI/WYKRESY/wykres_pazdziernik.png', 'Październik 2025')
]

for df, sciezka_mapy, sciezka_wykresu, nazwa in lista_danych:
    stworz_mape(df, stacje, sciezka_mapy)
    stworz_wykres_top_10(df, f"Top 10 tras - {nazwa}", sciezka_wykresu)